In [ ]:
import random
from pathlib import Path

import numpy as np
import tensorflow as tf
from solve.static_hypernet import Solver

In [ ]:
def set_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


def component_name(hyper_mode):
    return "static_hypernetwork" if hyper_mode else "baseline_conv"


def build_run_paths(dataset, model, hyper_mode=False, setting_name=None):
    run_name = f"{dataset}_{model}"
    if hyper_mode:
        run_name += "_hyper"
    if setting_name:
        run_name += f"_{setting_name}"
    run_root = Path("runs") / run_name
    return {
        "logpath": str(run_root / "logs"),
        "save_dir": str(run_root / "checkpoints"),
    }


def make_solver(dataset, model, hyper_mode=False, setting_name=None, **kwargs):
    paths = build_run_paths(dataset, model, hyper_mode=hyper_mode, setting_name=setting_name)
    return Solver(
        dataset=dataset,
        model=model,
        hyper_mode=hyper_mode,
        logpath=paths["logpath"],
        save_dir=paths["save_dir"],
        **kwargs,
    )


def evaluate_best(dataset, model, hyper_mode=False, setting_name=None, **kwargs):
    solver = make_solver(
        dataset,
        model,
        hyper_mode=hyper_mode,
        setting_name=setting_name,
        **kwargs,
    )
    metrics = solver.evaluate_best_checkpoint()
    test_loss, test_acc = metrics["test"]
    return {
        "dataset": dataset,
        "model": model,
        "component": component_name(hyper_mode),
        "setting": setting_name or "main",
        "max_epoch": kwargs.get("max_epoch", ""),
        "learning_rate": kwargs.get("learning_rate", ""),
        "batch_size": kwargs.get("batch_size", ""),
        "test_loss": test_loss,
        "test_acc": test_acc,
    }


def print_results(results):
    header = (
        f"{'dataset':<15} {'model':<12} {'component':<22} {'setting':<28} "
        f"{'epochs':>6} {'lr':>10} {'batch':>7} {'test_loss':>10} {'test_acc':>10}"
    )
    print(header)
    print("-" * len(header))
    for row in results:
        print(
            f"{row['dataset']:<15} {row['model']:<12} {row['component']:<22} "
            f"{row['setting']:<28} {str(row.get('max_epoch', '')):>6} "
            f"{str(row.get('learning_rate', '')):>10} {str(row.get('batch_size', '')):>7} "
            f"{row['test_loss']:>10.4f} {row['test_acc']:>10.4f}"
        )


set_random_seed(42)

In [ ]:
"""
Full Experiment Grid

Grid này bao phủ đầy đủ mọi tổ hợp dataset x model x hyper_mode:
4 dataset, 3 model, 2 component/mode = 24 run chính.
MNIST và CIFAR-10 là dataset theo paper; Fashion-MNIST và SVHN là 2 dataset mới.
"""

In [ ]:
DATASETS = ["mnist", "fashion_mnist", "cifar10", "svhn"]
MODELS = ["simplecnn", "wrn40_2", "resnet50"]
HYPER_MODES = [False, True]

MODEL_TRAIN_DEFAULTS = {
    "simplecnn": {"max_epoch": 20, "learning_rate": 5e-4, "batch_size": 1024},
    "wrn40_2": {"max_epoch": 10, "learning_rate": 5e-4, "batch_size": 512},
    "resnet50": {"max_epoch": 10, "learning_rate": 5e-4, "batch_size": 256},
}


def build_full_configs(setting_name="main", overrides=None):
    overrides = overrides or {}
    configs = []
    for dataset in DATASETS:
        for model in MODELS:
            for hyper_mode in HYPER_MODES:
                config = {
                    "dataset": dataset,
                    "model": model,
                    "hyper_mode": hyper_mode,
                    "setting_name": setting_name,
                    **MODEL_TRAIN_DEFAULTS[model],
                    **overrides,
                }
                configs.append(config)
    return configs


TRAIN_CONFIGS = build_full_configs(setting_name="main")
print(f"Total training configs: {len(TRAIN_CONFIGS)}")
TRAIN_CONFIGS[:4]

In [ ]:
"""
Train From Scratch

Cell dưới đây train toàn bộ 24 cấu hình chính từ scratch và lưu checkpoint riêng cho từng tổ hợp.
Mỗi run có thư mục riêng trong runs/{dataset}_{model} hoặc runs/{dataset}_{model}_hyper.
"""

In [ ]:
trained_solvers = {}
for config in TRAIN_CONFIGS:
    config = config.copy()
    setting_name = config.pop("setting_name")
    run_key = (config["dataset"], config["model"], config["hyper_mode"], setting_name)
    print("\n" + "=" * 100)
    print(
        f"Training dataset={config['dataset']} | model={config['model']} | "
        f"component={component_name(config['hyper_mode'])} | setting={setting_name}"
    )
    solver = make_solver(
        setting_name=None if setting_name == "main" else setting_name,
        show_sample=False,
        show_filters=False,
        **config,
    )
    solver.train()
    trained_solvers[run_key] = solver

In [ ]:
"""
Evaluation on Test Splits

Cell này nạp checkpoint tốt nhất của từng run chính và evaluate trên test split của tất cả dataset/model/hyper_mode đã train.
"""

In [ ]:
evaluation_results = []
for config in TRAIN_CONFIGS:
    config = config.copy()
    setting_name = config.pop("setting_name")
    result = evaluate_best(
        setting_name=None if setting_name == "main" else setting_name,
        **config,
    )
    evaluation_results.append(result)

print_results(evaluation_results)

In [ ]:
"""
Benchmark Hyperparameters and Components

Benchmark này vẫn bao phủ đầy đủ 24 tổ hợp dataset/model/hyper_mode, nhưng chạy thêm nhiều setting hyperparameter độc lập.
Các run benchmark được lưu trong runs/*_bench_* để không ghi đè checkpoint chính.
"""

In [ ]:
BENCHMARK_SETTINGS = [
    {
        "setting_name": "bench_lr5e4_default_bs",
        "overrides": {"max_epoch": 3, "learning_rate": 5e-4},
    },
    {
        "setting_name": "bench_lr1e3_bs256",
        "overrides": {"max_epoch": 3, "learning_rate": 1e-3, "batch_size": 256},
    },
]

BENCHMARK_CONFIGS = []
for setting in BENCHMARK_SETTINGS:
    BENCHMARK_CONFIGS.extend(
        build_full_configs(
            setting_name=setting["setting_name"],
            overrides=setting["overrides"],
        )
    )

print(f"Total benchmark configs: {len(BENCHMARK_CONFIGS)}")

In [ ]:
benchmark_results = []
for config in BENCHMARK_CONFIGS:
    config = config.copy()
    setting_name = config.pop("setting_name")
    print("\n" + "=" * 100)
    print(
        f"Benchmark dataset={config['dataset']} | model={config['model']} | "
        f"component={component_name(config['hyper_mode'])} | setting={setting_name}"
    )
    solver = make_solver(
        setting_name=setting_name,
        show_sample=False,
        show_filters=False,
        **config,
    )
    solver.train()
    metrics = solver.evaluate_best_checkpoint()
    test_loss, test_acc = metrics["test"]
    benchmark_results.append(
        {
            "dataset": config["dataset"],
            "model": config["model"],
            "component": component_name(config["hyper_mode"]),
            "setting": setting_name,
            "max_epoch": config["max_epoch"],
            "learning_rate": config["learning_rate"],
            "batch_size": config["batch_size"],
            "test_loss": test_loss,
            "test_acc": test_acc,
        }
    )

print_results(benchmark_results)

In [ ]:
expected_main_runs = len(DATASETS) * len(MODELS) * len(HYPER_MODES)
expected_benchmark_runs = expected_main_runs * len(BENCHMARK_SETTINGS)

assert len(TRAIN_CONFIGS) == expected_main_runs
assert len(BENCHMARK_CONFIGS) == expected_benchmark_runs

print(f"Main train/evaluation coverage: {len(TRAIN_CONFIGS)} / {expected_main_runs} configs")
print(f"Benchmark coverage: {len(BENCHMARK_CONFIGS)} / {expected_benchmark_runs} configs")

"""
Notes

- TRAIN_CONFIGS dùng để train và evaluate đầy đủ 24 tổ hợp chính.
- BENCHMARK_CONFIGS dùng để benchmark cùng 24 tổ hợp đó trên nhiều setting hyperparameter.
- Có thể tăng max_epoch trong MODEL_TRAIN_DEFAULTS hoặc BENCHMARK_SETTINGS khi cần kết quả cuối cùng cho báo cáo.
"""

In [ ]:
# Optional: save tabular results after running the evaluation and benchmark cells.
# Uncomment these lines when `evaluation_results` and `benchmark_results` are available.
# import json
# Path("runs").mkdir(exist_ok=True)
# Path("runs/evaluation_results.json").write_text(json.dumps(evaluation_results, indent=2), encoding="utf-8")
# Path("runs/benchmark_results.json").write_text(json.dumps(benchmark_results, indent=2), encoding="utf-8")